# Analisis Fondasi Data Energi HKUST dan Cuaca HKO

Notebook ini menyajikan fondasi analisis data untuk topik konsumsi energi kampus menggunakan smart meter HKUST dan data cuaca Hong Kong Observatory. Struktur notebook mengikuti alur OSEMN pada bagian yang sudah memiliki data final: Obtain, Scrub, dan Explore awal.

Fokus analisis saat ini adalah menyiapkan dataset harian T1440 yang bersih, terdokumentasi, memiliki konteks metadata, dan siap digunakan sebagai sumber Power BI. Bagian modelling anomaly detection dan interpretasi akhir tidak disajikan dalam notebook ini karena memerlukan output model final yang terpisah.

## Setup Analisis

Seluruh tabel yang digunakan di notebook ini berasal dari folder processed proyek. Notebook berperan sebagai ringkasan analitis, sedangkan proses parsing, cleaning, dan pembentukan datamart dilakukan melalui script pipeline.

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))

from config import PROCESSED_DIR, PROFILE_DIR, TARGET_BUILDING, SELECTED_METERS

## O - Obtain: Sumber Data dan Cakupan

Dataset utama berasal dari smart meter HKUST dengan interval harian T1440. Dataset pendukung berasal dari Hong Kong Observatory dan digunakan untuk memberikan konteks cuaca harian.

Cakupan utama analisis diarahkan pada meter aktif di `Cheng_Yu_Tung_Building` karena kumpulan meter ini memiliki coverage penuh, sinyal konsumsi aktif, dan metadata yang dapat dipetakan. Meter lain tetap dicatat dalam inventory agar kualitas data dan keputusan eksklusi dapat ditelusuri.

In [2]:
inventory = pd.read_csv(PROFILE_DIR / "t1440_meter_inventory_with_decision.csv")
weather = pd.read_csv(PROCESSED_DIR / "hko_weather_daily.csv", parse_dates=["date"])
obtain_summary = pd.DataFrame(
    [
        ["Jumlah meter T1440", int(inventory["meter_id"].nunique())],
        ["Jumlah meter subset utama", len(SELECTED_METERS)],
        ["Gedung fokus", TARGET_BUILDING],
        ["Jumlah hari cuaca HKO", len(weather)],
        ["Awal periode cuaca", weather["date"].min().date().isoformat()],
        ["Akhir periode cuaca", weather["date"].max().date().isoformat()],
    ],
    columns=["Metrik", "Nilai"],
)
display(obtain_summary)
display(inventory["quality_flag"].value_counts().rename_axis("quality_flag").reset_index(name="meter_count"))

,Metrik,Nilai
0,Jumlah meter T1440,26
1,Jumlah meter subset utama,12
2,Gedung fokus,Cheng_Yu_Tung_Building
3,Jumlah hari cuaca HKO,878
4,Awal periode cuaca,2022-01-01
5,Akhir periode cuaca,2024-05-27


,quality_flag,meter_count
0,active,18
1,zero_constant,4
2,near_zero_low_signal,3
3,short_coverage,1


## S - Scrub: Pembersihan dan Integrasi Data

Tahap Scrub menormalisasi tanggal, mengubah pembacaan meter menjadi konsumsi harian melalui differencing, menandai kasus kualitas data, dan menggabungkan konsumsi energi dengan cuaca harian berdasarkan tanggal.

Tabel akhir disusun sebagai star schema sederhana agar mudah digunakan di Power BI. Grain utama fact table adalah satu baris per tanggal dan meter terpilih.

In [3]:
dim_date = pd.read_csv(PROCESSED_DIR / "dim_date.csv", parse_dates=["date"])
dim_entity = pd.read_csv(PROCESSED_DIR / "dim_entity.csv")
dim_scenario = pd.read_csv(PROCESSED_DIR / "dim_scenario.csv")
fact = pd.read_csv(PROCESSED_DIR / "fact_energy_weather_daily.csv", parse_dates=["date"])

scrub_summary = pd.DataFrame(
    [
        ["Baris dim_date", len(dim_date)],
        ["Baris dim_entity", len(dim_entity)],
        ["Baris dim_scenario", len(dim_scenario)],
        ["Baris fact_energy_weather_daily", len(fact)],
        ["Jumlah entity di fact", int(fact["entity_id"].nunique())],
        ["Awal periode fact", fact["date"].min().date().isoformat()],
        ["Akhir periode fact", fact["date"].max().date().isoformat()],
        ["Baris model eligible", int(fact["is_model_eligible"].sum())],
    ],
    columns=["Metrik", "Nilai"],
)
display(scrub_summary)

,Metrik,Nilai
0,Baris dim_date,878
1,Baris dim_entity,26
2,Baris dim_scenario,3
3,Baris fact_energy_weather_daily,10524
4,Jumlah entity di fact,12
5,Awal periode fact,2022-01-02
6,Akhir periode fact,2024-05-27
7,Baris model eligible,10524


Validasi berikut memastikan bahwa tabel dimensi memiliki key unik dan fact table tidak memiliki duplikasi kombinasi tanggal dan entity.

In [4]:
checks = {
    "dim_date_unique": not dim_date.duplicated(["date"]).any(),
    "dim_entity_unique": not dim_entity.duplicated(["entity_id"]).any(),
    "dim_scenario_unique": not dim_scenario.duplicated(["scenario"]).any(),
    "fact_date_entity_unique": not fact.duplicated(["date", "entity_id"]).any(),
    "selected_meters_all_present": set(SELECTED_METERS).issubset(set(dim_entity["meter_id"])),
}
display(checks)
assert all(checks.values()), checks

{'dim_date_unique': True,
 'dim_entity_unique': True,
 'dim_scenario_unique': True,
 'fact_date_entity_unique': True,
 'selected_meters_all_present': True}

## E - Explore: Ringkasan Awal Konsumsi dan Kualitas Data

Eksplorasi awal dilakukan untuk membaca skala konsumsi, kontribusi meter, dan status kualitas data. Hasil pada bagian ini belum dimaksudkan sebagai interpretasi akhir, tetapi memberikan dasar yang kuat untuk visualisasi dan modelling berikutnya.

In [5]:
consumption_summary = fact["daily_consumption"].describe(percentiles=[0.25, 0.5, 0.75, 0.95]).to_frame("daily_consumption")
display(consumption_summary)

top_meter = (
    fact.groupby(["entity_id", "meter_name"], as_index=False)["daily_consumption"]
    .sum()
    .sort_values("daily_consumption", ascending=False)
)
top_meter["contribution_pct"] = top_meter["daily_consumption"] / top_meter["daily_consumption"].sum()
top_meter["cumulative_contribution_pct"] = top_meter["contribution_pct"].cumsum()
display(top_meter.head(12))

,daily_consumption
count,10524.000000
mean,1300.521570
std,897.735631
min,110.000000
25%,700.000000
50%,980.000000
75%,1980.000000
95%,3160.000000
max,4100.000000


,entity_id,meter_name,daily_consumption,contribution_pct,cumulative_contribution_pct
1,D0849,Meter_D0849,2822400.0,0.206215,0.206215
0,D0848,Meter_D0848,2053420.0,0.150030,0.356245
6,D0854,Meter_D0854,1866900.0,0.136403,0.492648
5,D0853,Meter_D0853,1774050.0,0.129619,0.622267
10,D0862,Meter_D0862,898620.0,0.065656,0.687923
7,D0857,Meter_D0857,877200.0,0.064091,0.752015
3,D0851,Meter_D0851,855730.0,0.062523,0.814537
9,D0861,Meter_D0861,760510.0,0.055566,0.870103
4,D0852,Meter_D0852,679530.0,0.049649,0.919752
8,D0860,Meter_D0860,607590.0,0.044393,0.964145


Ringkasan kualitas data menunjukkan bahwa meter bermasalah tetap terdokumentasi dalam dimensi entity, sedangkan fact utama hanya berisi meter aktif terpilih. Nilai cuaca yang hilang dipertahankan pada kolom asli dan disediakan kolom model-safe untuk kebutuhan analisis lanjutan.

In [6]:
data_quality = pd.read_csv(PROCESSED_DIR / "data_quality_summary.csv")
display(data_quality)
display(fact["data_quality_flag"].value_counts().rename_axis("data_quality_flag").reset_index(name="rows"))

,summary_group,metric,value,notes
0,meter_quality_flag,active,18,Count of T1440 meters by quality flag.
1,meter_quality_flag,near_zero_low_signal,3,Count of T1440 meters by quality flag.
2,meter_quality_flag,short_coverage,1,Count of T1440 meters by quality flag.
3,meter_quality_flag,zero_constant,4,Count of T1440 meters by quality flag.
4,selection_decision,comparison_or_secondary,4,Count of T1440 meters by selection decision.
5,selection_decision,exclude_from_main_model,10,Count of T1440 meters by selection decision.
6,selection_decision,selected_main_subset,12,Count of T1440 meters by selection decision.
7,weather_missing_days,rainfall_mm,5,Missing HKO rainfall days in project period.
8,weather_missing_days,global_solar_radiation_mj_m2,1,Missing HKO solar radiation days in project pe...
9,fact_rows,fact_energy_weather_daily,10524,Rows in final Date x Entity fact table.


,data_quality_flag,rows
0,valid,10452
1,weather_imputed,72


## Kesiapan Data untuk Power BI

Tabel berikut merupakan dataset yang dapat diimpor ke Power BI Desktop. Relasi utama menghubungkan fact table ke `dim_date`, `dim_entity`, dan `dim_scenario`.

In [7]:
outputs = [
    "fact_energy_weather_daily.csv",
    "dim_date.csv",
    "dim_entity.csv",
    "dim_scenario.csv",
    "data_quality_summary.csv",
    "data_dictionary_energy_dashboard.csv",
]
pd.DataFrame({
    "file": outputs,
    "exists": [(PROCESSED_DIR / name).exists() for name in outputs],
    "path": [str(PROCESSED_DIR / name) for name in outputs],
})

,file,exists,path
0,fact_energy_weather_daily.csv,True,C:\Users\Khalfani Shaquille\Documents\GitHub\K...
1,dim_date.csv,True,C:\Users\Khalfani Shaquille\Documents\GitHub\K...
2,dim_entity.csv,True,C:\Users\Khalfani Shaquille\Documents\GitHub\K...
3,dim_scenario.csv,True,C:\Users\Khalfani Shaquille\Documents\GitHub\K...
4,data_quality_summary.csv,True,C:\Users\Khalfani Shaquille\Documents\GitHub\K...
5,data_dictionary_energy_dashboard.csv,True,C:\Users\Khalfani Shaquille\Documents\GitHub\K...


## Catatan Keterbatasan

Dataset T1440 harian hanya mencakup 26 meter, dan fact utama pada notebook ini menggunakan 12 meter aktif dari satu gedung. Karena itu, hasil analisis pada notebook ini sebaiknya dibaca sebagai fondasi analisis meter terpilih, bukan klaim representatif untuk seluruh kampus. Data cuaca digunakan sebagai konteks pendukung dan tidak boleh ditafsirkan sebagai penyebab tunggal perubahan konsumsi.